# PoC: Page Replacement with Tabular Q-Learning

Compares **FIFO**, **LRU**, **Optimal** and a **Q-learning agent** on four synthetic workloads.

**RL formulation used in this notebook**
- **Candidates:** the 3 least-recently-used resident pages (keeps the action space small)
- **State:** access-frequency bucket (0, 1, 2+) of each candidate, giving 27 states
- **Action:** 0, 1 or 2, meaning evict the 1st, 2nd or 3rd least-recently-used candidate
- **Reward (training only):** +1 if the evicted page is not requested again within the next W requests, -1 if it is. This uses the trace offline, so at test time the agent uses no future knowledge.
- **Algorithm:** Q-learning with epsilon-greedy exploration, updated on eviction events

In [ ]:
import random
from collections import OrderedDict, defaultdict, deque
import matplotlib.pyplot as plt

%matplotlib inline

NUM_PAGES = 30
TRACE_LEN = 4000

## 1. Workload generators

In [ ]:
def loop_trace(k, n=TRACE_LEN, seed=0):
    """Cyclic scan over k+2 pages (worst case for LRU/FIFO)."""
    r = random.Random(seed)
    span = k + 2
    t = []
    while len(t) < n:
        t.extend(range(span))
        if r.random() < 0.02:               # small noise
            t.append(r.randrange(NUM_PAGES))
    return t[:n]


def locality_trace(k, n=TRACE_LEN, seed=0):
    """Working set of k pages that drifts over time."""
    r = random.Random(seed)
    ws = r.sample(range(NUM_PAGES), k)
    t = []
    for i in range(n):
        if i % 300 == 0:                    # phase change
            ws = r.sample(range(NUM_PAGES), k)
        t.append(r.choice(ws) if r.random() < 0.9 else r.randrange(NUM_PAGES))
    return t


def zipf_trace(k, n=TRACE_LEN, seed=0):
    """Skewed popularity: few hot pages, long cold tail."""
    r = random.Random(seed)
    w = [1 / (i + 1) for i in range(NUM_PAGES)]
    return r.choices(range(NUM_PAGES), weights=w, k=n)


def mixed_trace(k, n=TRACE_LEN, seed=0):
    """Hot pages plus periodic one-time scans (cache pollution)."""
    r = random.Random(seed)
    hot = list(range(max(2, k - 1)))
    t, scan_page = [], k
    while len(t) < n:
        t.extend(r.choice(hot) for _ in range(r.randint(10, 25)))
        for _ in range(r.randint(3, 8)):    # scan of mostly-new pages
            t.append(scan_page)
            scan_page = scan_page + 1 if scan_page < NUM_PAGES - 1 else k
    return t[:n]


WORKLOADS = {"Loop": loop_trace, "Locality": locality_trace,
             "Zipf": zipf_trace, "Hot+Scan": mixed_trace}

## 2. Baseline algorithms (FIFO, LRU, Optimal)

In [ ]:
def fifo(trace, k):
    mem, q, faults = set(), deque(), 0
    for p in trace:
        if p in mem:
            continue
        faults += 1
        if len(mem) == k:
            mem.remove(q.popleft())
        mem.add(p)
        q.append(p)
    return faults


def lru(trace, k):
    mem, faults = OrderedDict(), 0
    for p in trace:
        if p in mem:
            mem.move_to_end(p)
            continue
        faults += 1
        if len(mem) == k:
            mem.popitem(last=False)
        mem[p] = True
    return faults


def optimal(trace, k):
    mem, faults = set(), 0
    for i, p in enumerate(trace):
        if p in mem:
            continue
        faults += 1
        if len(mem) == k:
            def next_use(x):
                try:
                    return trace.index(x, i + 1)
                except ValueError:
                    return float("inf")
            mem.remove(max(mem, key=next_use))
        mem.add(p)
    return faults

## 3. Q-learning agent

In [ ]:
class QAgent:
    N_CAND = 3

    def __init__(self, alpha=0.1, gamma=0.5, eps=0.2, window=None):
        self.Q = defaultdict(lambda: [0.0] * self.N_CAND)
        self.alpha, self.gamma, self.eps, self.window = alpha, gamma, eps, window

    @staticmethod
    def _bucket(c):
        return min(c, 2)

    def _state(self, cands, freq):
        return tuple(self._bucket(freq[c]) for c in cands)

    def run(self, trace, k, train=False, seed=0):
        rng = random.Random(seed)
        W = self.window or k
        mem = OrderedDict()                 # LRU order: first = least recent
        freq = {}
        faults = 0
        prev = None                         # (state, action, reward) awaiting next state
        for i, p in enumerate(trace):
            if p in mem:
                mem.move_to_end(p)
                freq[p] += 1
                continue
            faults += 1
            if len(mem) == k:
                cands = list(mem.keys())[: self.N_CAND]
                s = self._state(cands, freq)
                if prev is not None and train:      # Q-learning update for previous step
                    ps, pa, pr = prev
                    td = pr + self.gamma * max(self.Q[s]) - self.Q[ps][pa]
                    self.Q[ps][pa] += self.alpha * td
                if train and rng.random() < self.eps:
                    a = rng.randrange(len(cands))
                else:
                    qs = self.Q[s][: len(cands)]
                    a = qs.index(max(qs))
                victim = cands[a]
                if train:
                    reward = -1.0 if victim in trace[i + 1: i + 1 + W] else 1.0
                    prev = (s, a, reward)
                del mem[victim]
                del freq[victim]
            mem[p] = True
            freq[p] = 1
        return faults


def train_agent(k, wl_fn, episodes=60):
    agent = QAgent()
    for ep in range(episodes):
        agent.eps = max(0.02, 0.3 * (1 - ep / episodes))
        agent.run(wl_fn(k, seed=1000 + ep), k, train=True, seed=ep)
    return agent

## 4. Run experiments
Train on generated traces, test on an unseen trace (seed 42).

In [ ]:
ks = [3, 4, 5, 6, 8]
results = {}                                # (workload, k) -> {algo: faults}

for name, fn in WORKLOADS.items():
    for k in ks:
        agent = train_agent(k, fn)
        test = fn(k, seed=42)               # unseen trace
        results[(name, k)] = {
            "FIFO": fifo(test, k), "LRU": lru(test, k),
            "RL": agent.run(test, k), "OPT": optimal(test, k),
        }

print(f"{'Workload':10}{'Frames':>7}{'FIFO':>8}{'LRU':>8}{'RL':>8}{'OPT':>8}")
for (name, k), r in results.items():
    print(f"{name:10}{k:>7}{r['FIFO']:>8}{r['LRU']:>8}{r['RL']:>8}{r['OPT']:>8}")

## 5. Page faults vs number of frames

In [ ]:
fig, axes = plt.subplots(1, 4, figsize=(18, 4))
for ax, name in zip(axes, WORKLOADS):
    for algo, style in [("FIFO", "s--"), ("LRU", "o-"), ("RL", "^-"), ("OPT", "k:")]:
        ax.plot(ks, [results[(name, k)][algo] for k in ks], style, label=algo)
    ax.set_title(name)
    ax.set_xlabel("Number of frames")
    ax.set_ylabel("Page faults")
    ax.legend()
plt.tight_layout()
plt.savefig("page_faults_comparison.png", dpi=130)
plt.show()

## 6. Hit ratio (optional extra)
Hit ratio = 1 - faults / trace length. Useful for your own analysis.

In [ ]:
print(f"{'Workload':10}{'Frames':>7}{'FIFO':>8}{'LRU':>8}{'RL':>8}{'OPT':>8}   (hit ratio)")
for (name, k), r in results.items():
    hr = {a: 1 - f / TRACE_LEN for a, f in r.items()}
    print(f"{name:10}{k:>7}{hr['FIFO']:>8.3f}{hr['LRU']:>8.3f}{hr['RL']:>8.3f}{hr['OPT']:>8.3f}")